# 2. Анализ обучения\n\nВизуализация кривых обучения, функций потерь и метрик по эпохам.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)

PROJECT_ROOT = Path('..').resolve()
# Укажите имя эксперимента
EXP_NAME = 'fire_v1'
RESULTS_DIR = PROJECT_ROOT / 'runs' / 'detect' / EXP_NAME

In [ ]:
# Загрузка results.csv (генерируется ultralytics автоматически)
csv_path = RESULTS_DIR / 'results.csv'
if csv_path.exists():
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()
    print(f"Загружено {len(df)} эпох")
    print(f"Колонки: {list(df.columns)}")
    df.head()
else:
    print(f"Файл не найден: {csv_path}")
    print("Сначала запустите обучение: python src/train.py")

## 2.1 Кривые потерь

In [ ]:
if csv_path.exists():
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    loss_cols = {
        'train/box_loss': 'Box Loss',
        'train/cls_loss': 'Cls Loss', 
        'train/dfl_loss': 'DFL Loss',
    }
    val_loss_cols = {
        'val/box_loss': 'Box Loss',
        'val/cls_loss': 'Cls Loss',
        'val/dfl_loss': 'DFL Loss',
    }
    
    for idx, (train_col, name) in enumerate(loss_cols.items()):
        val_col = train_col.replace('train/', 'val/')
        if train_col in df.columns:
            axes[idx].plot(df['epoch'], df[train_col], label='Train', linewidth=2)
        if val_col in df.columns:
            axes[idx].plot(df['epoch'], df[val_col], label='Val', linewidth=2, linestyle='--')
        axes[idx].set_title(name)
        axes[idx].set_xlabel('Epoch')
        axes[idx].set_ylabel('Loss')
        axes[idx].legend()
    
    plt.suptitle('Кривые потерь', fontsize=14)
    plt.tight_layout()
    plt.savefig(str(RESULTS_DIR / 'loss_curves.png'), dpi=150, bbox_inches='tight')
    plt.show()

## 2.2 Метрики по эпохам (mAP, Precision, Recall)

In [ ]:
if csv_path.exists():
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    metric_groups = [
        (['metrics/precision(B)', 'metrics/recall(B)'], ['Precision', 'Recall'], 'Precision & Recall'),
        (['metrics/mAP50(B)'], ['mAP@0.5'], 'mAP@0.5'),
        (['metrics/mAP50-95(B)'], ['mAP@0.5:0.95'], 'mAP@0.5:0.95'),
    ]
    
    for idx, (cols, labels, title) in enumerate(metric_groups):
        for col, label in zip(cols, labels):
            if col in df.columns:
                axes[idx].plot(df['epoch'], df[col], label=label, linewidth=2)
        axes[idx].set_title(title)
        axes[idx].set_xlabel('Epoch')
        axes[idx].set_ylabel('Metric')
        axes[idx].legend()
        axes[idx].set_ylim(0, 1)
    
    plt.suptitle('Метрики валидации по эпохам', fontsize=14)
    plt.tight_layout()
    plt.savefig(str(RESULTS_DIR / 'metrics_curves.png'), dpi=150, bbox_inches='tight')
    plt.show()
    
    # Лучшие метрики
    if 'metrics/mAP50(B)' in df.columns:
        best_epoch = df['metrics/mAP50(B)'].idxmax()
        print(f"\nЛучшая эпоха: {int(df.loc[best_epoch, 'epoch'])}")
        print(f"  mAP@0.5:      {df.loc[best_epoch, 'metrics/mAP50(B)']:.4f}")
        if 'metrics/mAP50-95(B)' in df.columns:
            print(f"  mAP@0.5:0.95: {df.loc[best_epoch, 'metrics/mAP50-95(B)']:.4f}")
        if 'metrics/precision(B)' in df.columns:
            print(f"  Precision:    {df.loc[best_epoch, 'metrics/precision(B)']:.4f}")
        if 'metrics/recall(B)' in df.columns:
            print(f"  Recall:       {df.loc[best_epoch, 'metrics/recall(B)']:.4f}")